# Model Comparison Script

In [1]:
# Enter or regenerate the metrics from the two performance notebooks.
# The values below are calculated by fitting the same reproducible pipelines.
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score

DATA_PATH = Path('engineered_dataset.csv')

# If files are loaded in repository layout hide above and unmark below:
# DATA_PATH = Path('/.../.../Datasets/Engineered/engineered_dataset.csv')

df = pd.read_csv(DATA_PATH)
X=df.drop(columns=['returned','order_id']); 
y=df['returned'].astype(int)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)

num=X.select_dtypes(include=['number']).columns.tolist(); 
cat=X.select_dtypes(exclude=['number']).columns.tolist()
pre_lr=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),num),
                          ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)])
pre_rf=ColumnTransformer([('num',SimpleImputer(strategy='median'),num),
                          ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)])
lr=Pipeline([('preprocessor',pre_lr),('classifier',LogisticRegression(C=1.0,penalty='l2',solver='liblinear',class_weight='balanced',max_iter=1000,random_state=42))])
rf=Pipeline([('preprocessor',pre_rf),('classifier',RandomForestClassifier(n_estimators=300,max_features='sqrt',class_weight='balanced',random_state=42,n_jobs=-1))])
rows=[]
for name,m in [('Logistic Regression',lr),('Random Forest',rf)]:
    m.fit(Xtr,ytr); p=m.predict(Xte); pr=m.predict_proba(Xte)[:,1]
    rows.append({'Model':name,'Accuracy':accuracy_score(yte,p),'Precision':precision_score(yte,p,zero_division=0),'Recall':recall_score(yte,p,zero_division=0),'F1-score':f1_score(yte,p,zero_division=0),'ROC-AUC':roc_auc_score(yte,pr)})
comparison=pd.DataFrame(rows).set_index('Model')
comparison

,Accuracy,Precision,Recall,F1-score,ROC-AUC
Model,,,,,
Logistic Regression,0.565750,0.540861,0.562684,0.551557,0.590093
Random Forest,0.558075,0.544696,0.419511,0.473977,0.579232
